In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np
import pycav

#### Basic `Beam()`

In [ ]:
b = pycav.Beam(1030e-9, step=0.0001)
b.set_w0([0.5e-3, 0.5e-3])
b.add_space(0.01)
b.add_kerr_medium(1.75, 2.6e-20, 1e-3, 200000e6, 25, 0.001, "KL")
b.add_space(0.02)

d, ww = b.get_beam()
ws, wt = ww.T

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2))
ax.plot(d, ws)
ax.plot(d, wt)

#### Basic `Cavity()`

In [ ]:
cwl = 1030e-9
fig, ax = plt.subplots(figsize=(8, 2))

NN = 25
for i in range(1, NN):
    c = pycav.Cavity(cwl)
    c.add_mirror(1, idn="M1")
    c.add_space(0.02, idn="L1")
    c.add_kerr_medium(1.75, 2.6e-20, 1e-3, 200e6, kerrit=i, idn="KL")
    c.add_space(0.02, idn="L2")
    c.add_mirror(1, idn="M2")
    d, ww = c.get_beam()
    ws, wt = ww.T
    ax.plot(d, ws, lw=1, color=plt.cm.viridis((i - 1) / (NN - 2)))

**Note:** The Kerr lens depends on the beam size in the Kerr medium, which in turn depends on the Kerr lens. pycav solves this iteratively by ramping up the power in `kerrit` steps (default: 25). The plot above shows how the beam converges with the number of iterations; check the convergence for your own cavity.

#### same `Cavity()` with the `CavityDesigner()`

In [ ]:
varpars = [
    ["KL", "P", np.linspace(100e6, 300e6, 11)],
    ["L1", "d", np.linspace(0, 0.1, 101)],
    ["L2", "d", np.linspace(0, 0.1, 101)],
]
elsofin = ["M1", "M2"]
cd = pycav.CavityDesigner(c, varpars, elsofin)

**Note:** Close to the stability limits, the result can depend strongly on `kerrit`. Increase it if the beam sizes change noticeably with the number of iterations.

#### Kerr-lens mode-locked thin-disk cavity

In [ ]:
c2 = pycav.Cavity(cwl)
fk, l1, l2 = 0.124048, 0.4917, 0.427934
c2.add_mirror(idn="OC")
c2.add_space(1.83)
c2.add_mirror(20, np.deg2rad(2))
c2.add_space(3.33 - 1.83)
c2.add_mirror(20, np.deg2rad(2))
c2.add_space(6.6 - 3.3)
c2.add_mirror(0.9)
c2.add_space(l1, idn="l1")
c2.add_kerr_medium(1.75, 2.6e-20, 1e-3, 20e6, idn="KL")
c2.add_space(l2, idn="l2")
c2.add_mirror(0.9, np.deg2rad(-2))
c2.add_space(7.87 - 7.55)
c2.add_mirror(idn="end")
dd2, ww2 = c2.get_beam()
ws2, wt2 = ww2.T

In [ ]:
varpars = [
    ("KL", "P", np.linspace(0, 1e9, 100)),
    ("l1", "d", np.linspace(0.4, 0.6, 100)),
    ("l2", "d", np.linspace(0.4, 0.6, 100)),
]
elsofin = ("OC", "end")
cd = pycav.CavityDesigner(c2, varpars, elsofin, figsize=(8, 3))